### Data Transformation in Silver Layer

#### Read the bronze layer

In [0]:
df=spark.read.format('delta').load('/Volumes/workspace/default/practice/SuperStore_Sales/Bronze')
display(df)

#### Inspect the data

In [0]:
df.printSchema()

#### Import types and functions

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

#### casting the type of sales column after removing thousand separator

In [0]:
df= df.withColumn('sales',regexp_replace(col('sales'),",","").cast(DoubleType()))
display(df)

In [0]:
df.printSchema()

In [0]:
df.select([ sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

#### no missing values
#### Remove duplicates

In [0]:
df.groupBy(df.columns).agg(count("*").alias('count')).filter(col('count')>1).show()
# here is no duplicated columns
#if there is any duplicates df.dropDuplicates()/d.drpDuplicates(["id"])


In [0]:
df.limit(10).display()

#### check unique values for all columns

In [0]:
for c in df.columns:
    print(f"Unique values in column '{c}':")
    df.select(c).distinct().show(truncate=False)

#### removing training spaces in the data 

In [0]:
# for c, t in df.dtypes:
#     if t=='string':
#         print(c)

string_cols=[c for c,t in df.dtypes if t=='string']

for cols in string_cols:
    df_clean=df_clean.withColumn(cols,trim(col(cols)))
    df_clean=df_clean.withColumn(cols,regexp_replace(col(cols),"'", ""))
display(df_clean)



In [0]:
df_clean.select('region').distinct().show()


#### Standardize the data

In [0]:
# replace the data if any abbrivations with their full name

In [0]:
df_clean=df_clean.withColumn('processed_at', current_date())

In [0]:
df_clean.write.format('delta').mode('overwrite').option('overwriteSchema',True).save('/Volumes/workspace/default/practice/SuperStore_Sales/Silver')